# Natural language to SQL

**Run in [Google Colab](https://colab.research.google.com/) For GPU.**

This model have  Mistral as a base and it has been fine-tuned to excel in SQL code generation.

In [9]:
from google.colab import userdata
#userdata.get('HF_TOKEN')

In [10]:
#Install the lastest versions of peft & transformers library recommended
#if you want to work with the most recent models
!pip install -q git+https://github.com/huggingface/peft.git
!pip install git+https://github.com/huggingface/accelerate.git
!pip install git+https://github.com/huggingface/transformers.git
!pip install bitsandbytes

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 76, in resolve
    collected = self.factory.collect_root_requirements(root_reqs)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/factory.py", line 538, in collect_root_requirements
    reqs = list(
           ^^^^^
  File "/usr

In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import accelerate

In [12]:
model_name = "defog/sqlcoder-7b"

We need to create the Quantization configuration to load the Model.

It is a large model and I want it to fit in a 16GB GPU, I'm going to use a 4 bits quantization.

If you want to learn more about quantization, refer to this article: [QLoRA: Training a Large Language Model on a 16GB GPU.](https://medium.com/towards-artificial-intelligence/qlora-training-a-large-language-model-on-a-16gb-gpu-00ea965667c1)

You can try to use this model in a 8 bit quantizations and check in you see any improvements in the results.

In [13]:
bnb_config = BitsAndBytesConfig(
  load_in_4bit=True,
  bnb_4bit_use_double_quant=True,
  bnb_4bit_quant_type="nf4",
  bnb_4bit_compute_dtype=torch.bfloat16
)


To load the model I pass to the AutoModelForCasualLM teh quantization configurations, and HuggingFace take care of all the hard work.

In [14]:
foundation_model = AutoModelForCausalLM.from_pretrained(model_name,
                    quantization_config=bnb_config,
                    device_map='auto',
                    use_cache = True)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [15]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
eos_token_id = tokenizer.convert_tokens_to_ids(["```"])[0]

tokenizer_config.json:   0%|          | 0.00/915 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

This function wraps the call to *model.generate*

In [16]:
#this function returns the outputs from the model received, and inputs.
def get_outputs(model, inputs, max_new_tokens=400):
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        num_return_sequences=1,
        eos_token_id=eos_token_id,
        pad_token_id=eos_token_id,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=5
    )
    return outputs

# Prompt without Shots.
In this first PROMPT we are going to give Instructions to the model and pass the structure of the Database.

The instructions are significantly different from those we are passing to GPT-3.5-Turbo. This model is really well fine-tuned, but it is smaller than GPT-3.5.

We need to be more clear with the instructions, as it does not have the same capacity to understand our orders as GPT-3.5.

In [36]:
sp_nl2sql = """
    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question

    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    CREATE TABLE employees (
        ID_usr INT,
        name VARCHAR(100)
    );

    CREATE TABLE salary (
        ID_usr INT,
        salary INT,
        year INT
    );

    CREATE TABLE studies (
        ID_study INT,
        ID_usr INT,
        Institution VARCHAR(100),
        degree VARCHAR(100)
    );

    ### Response
    Based on your instructions, here is the SQL query I have generated to answer the question
    `{question}`:
    ```sql3
    """

In [37]:
sp_nl2sql = sp_nl2sql.format(question="Return the name of the best paid employee")
print(sp_nl2sql)


    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question

    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    CREATE TABLE employees (
        ID_usr INT,
        name VARCHAR(100)
    );

    CREATE TABLE salary (
        ID_usr INT,
        salary INT,
        year INT
    );

    CREATE TABLE studies (
        ID_study INT,
        ID_usr INT,
        Institution VARCHAR(100),
        degree VARCHAR(100)
    );

    ### Response
    Based on your instructions, here is the SQL query I have generated to answer the question
    `Return the name of the best paid employee`:
    ```sql3
    


In [38]:
input_sentences = tokenizer(sp_nl2sql, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)

In [39]:
#Empty the cache in orde to do more calls without problems.
torch.cuda.empty_cache()

In [40]:
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

SELECT employees.name FROM employees JOIN (SELECT salary.ID_usr, MAX(salary.salary) AS max_salary FROM salary GROUP BY salary.ID_usr) AS subquery ON employees.ID_usr = subquery.ID_usr WHERE subquery.max_salary = (SELECT MAX(max_salary) FROM subquery);


The SQL Order is correct.

#Prompt with shots OpenAI Style.
In this second prompt we are going to add some Shots with samples to see if our SQL style affects the model.

In [41]:
sp_nl2sql2 = """
    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL In the ### Samples section to clearn more about teh Databases structure


    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    CREATE TABLE employees (
        ID_usr INT,
        name VARCHAR(100)
    );

    CREATE TABLE salary (
        ID_usr INT,
        salary INT,
        year INT
    );

    CREATE TABLE studies (
        ID_study INT,
        ID_usr INT,
        Institution VARCHAR(100),
        degree VARCHAR(100)
    );

    ### Response
    YOUR QERIES AND SAMPLE RESPONSES HERE

    `{question}`:
    ```sql3
    """


In [42]:
sp_nl2sql2 = sp_nl2sql2.format(question="Return The name of the best paid employee")
(print(sp_nl2sql2))


    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL In the ### Samples section to clearn more about teh Databases structure


    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    CREATE TABLE employees (
        ID_usr INT,
        name VARCHAR(100)
    );

    CREATE TABLE salary (
        ID_usr INT,
        salary INT,
        year INT
    );

    CREATE TABLE studies (
        ID_study INT,
        ID_usr INT,
        Institution VARCHAR(100),
        degree VARCHAR(100)
    );

    ### Response
    YOUR QERIES AND SAMPLE RESPONSES HERE

    `Return The name of the best paid employee`:
    ```sql3
    


In [44]:
input_sentences = tokenizer(sp_nl2sql2, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()

In [45]:
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

SELECT AVG(salary.salary) AS average_salary FROM employees JOIN salary ON employees.id_usr = salary.id_usr WHERE;


The Order is really different from the one obtained with the first prompt.

The first difference is the format. But The SQL is realy more simple, at least it is my sensation.

#Prompt with Shots in Sample Style.

In this prompt, we will place the examples in a separate section, and in the instructions, we will instruct the model to pay attention to them in order to generate the SQL commands.

In [46]:
sp_nl2sql3b = """
    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL In the ### Samples section to learn more about the Databases structure


    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    CREATE TABLE employees (
        ID_usr INT,
        name VARCHAR(100)
    );

    CREATE TABLE salary (
        ID_usr INT,
        salary INT,
        year INT
    );

    CREATE TABLE studies (
        ID_study INT,
        ID_usr INT,
        Institution VARCHAR(100),
        degree VARCHAR(100)
    );

    ### Samples


    Question: Return all employee names.
    ```sql3
    SELECT name
    FROM employees;
    ```

    Question: Return employee names and salaries.
    ```sql3
    SELECT e.name, s.salary
    FROM employees e
    JOIN salary s
        ON e.ID_usr = s.ID_usr;
    ```

    Question: Return employee names and institutions.
    ```sql3
    SELECT e.name, st.Institution
    FROM employees e
    JOIN studies st
        ON e.ID_usr = st.ID_usr;

    ### Response
    Based on your instructions, here is the SQL query I have generated to answer the question
    `{question}`:
    ```sql3
    """


In [47]:
sp_nl2sql3 = sp_nl2sql3b.format(question="Return The name of the best paid employee")
print (sp_nl2sql3)


    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL In the ### Samples section to learn more about the Databases structure


    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    CREATE TABLE employees (
        ID_usr INT,
        name VARCHAR(100)
    );

    CREATE TABLE salary (
        ID_usr INT,
        salary INT,
        year INT
    );

    CREATE TABLE studies (
        ID_study INT,
        ID_usr INT,
        Institution VARCHAR(100),
        degree VARCHAR(100)
    );
    
    ### Samples
    
    
    Question: Return all employee names.
    ```sql3
    SELECT name
    FROM employees;
    ```

    Question: Return employee names and salaries.
    ```sql3
    SEL

In [48]:
input_sentences = tokenizer(sp_nl2sql3, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()

In [49]:
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

SELECT e.name
    FROM employees e
    JOIN (SELECT s.ID_usr, MAX(s.salary) AS max_salary
    FROM salary s
    GROUP BY s.ID_usr) AS max_salary_per_user
        ON e.ID_usr = max_salary_per_user.ID_usr;


#Now the question in spanish.


In [50]:
sp_nl2sql3 = sp_nl2sql3b.format(question="Devuelve el nombre del empleado mejor pagado")
print (sp_nl2sql3)


    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL In the ### Samples section to learn more about the Databases structure


    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    CREATE TABLE employees (
        ID_usr INT,
        name VARCHAR(100)
    );

    CREATE TABLE salary (
        ID_usr INT,
        salary INT,
        year INT
    );

    CREATE TABLE studies (
        ID_study INT,
        ID_usr INT,
        Institution VARCHAR(100),
        degree VARCHAR(100)
    );
    
    ### Samples
    
    
    Question: Return all employee names.
    ```sql3
    SELECT name
    FROM employees;
    ```

    Question: Return employee names and salaries.
    ```sql3
    SEL

In [51]:
input_sentences = tokenizer(sp_nl2sql3, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()

In [52]:
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

SELECT e.name, MAX(s.salary) AS max_salary
    FROM employees e
    JOIN salary s
        ON e.ID_usr = s.ID_usr
    GROUP BY e.name
    ORDER BY max_salary DESC NULLS LAST;


The generated SQL command is the same regardless of where we have placed the examples.

# Conclusions

In this exercise, I explored how different prompt engineering strategies affect SQL generation using the `defog/sqlcoder-7b` model. I tested multiple prompt styles, including zero-shot prompting, prompts with inline examples, prompts with a dedicated samples section, and a multilingual version using Spanish.

The experiments showed that prompt structure has a significant impact on the quality of the generated SQL. The zero-shot prompt produced more complex SQL and even hallucinated invalid logic by referencing a derived table alias incorrectly. This demonstrated that the model can understand the task but may struggle with reliability when no examples are provided.

## SQL Result — Prompt Without Shots

```sql
SELECT employees.name
FROM employees
JOIN (
    SELECT salary.ID_usr, MAX(salary.salary) AS max_salary
    FROM salary
    GROUP BY salary.ID_usr
) AS subquery
    ON employees.ID_usr = subquery.ID_usr
WHERE subquery.max_salary = (
    SELECT MAX(max_salary)
    FROM subquery
);
```

This query attempted to solve the problem correctly, but it referenced the alias subquery inside another subquery, which is invalid in many SQL systems.

Adding examples improved the outputs noticeably. The prompts with shots generated cleaner and more readable SQL queries, and the model followed the style of the examples closely. However, one of the prompts with examples still failed by generating an unrelated query using AVG(salary) and an invalid WHERE; clause. This showed that few-shot prompting improves performance but does not completely eliminate hallucinations or logical mistakes.

## SQL Result — Prompt With Shots OpenAI Style

```sql
SELECT AVG(salary.salary) AS average_salary
FROM employees
JOIN salary
    ON employees.id_usr = salary.id_usr
WHERE;
```
This query was incorrect because it answered a different question and also produced invalid SQL syntax.

The prompt using a dedicated ### Samples section produced the most stable and syntactically correct outputs. The generated SQL was logically closer to the expected answer and demonstrated better understanding of joins and aggregations. This suggests that separating examples into their own section helps the model better distinguish between instructions, examples, and the actual task.

## SQL Result — Prompt With Samples Section

```sql
SELECT e.name
FROM employees e
JOIN (
    SELECT s.ID_usr, MAX(s.salary) AS max_salary
    FROM salary s
    GROUP BY s.ID_usr
) AS max_salary_per_user
    ON e.ID_usr = max_salary_per_user.ID_usr;
```
This query was syntactically valid and cleaner than the previous versions, but it did not fully answer the question because it returned all employees instead of only the best paid employee.

The Spanish version produced surprisingly strong results. The model correctly generated a query using MAX(), GROUP BY, and ORDER BY. Although it missed LIMIT 1, the SQL logic was still accurate.

## SQL Result — Spanish Prompt

```sql
SELECT e.name, MAX(s.salary) AS max_salary
FROM employees e
JOIN salary s
    ON e.ID_usr = s.ID_usr
GROUP BY e.name
ORDER BY max_salary DESC NULLS LAST;
```
This was one of the best results because the joins, aggregation, and ordering were all correct.

One important lesson from this exercise is that smaller fine-tuned models are highly sensitive to prompt clarity, formatting, and examples. Typos, unclear instructions, or poorly designed samples can quickly reduce output quality. I also learned that examples strongly influence the SQL style and structure produced by the model, which means prompt design is extremely important in natural-language-to-SQL systems.

Overall, the model demonstrated strong SQL generation capabilities, especially when guided with clear instructions and high-quality examples. Few-shot prompting and structured sample sections provided the best balance between correctness, readability, and consistency.

# Exercise
 - Complete the prompts similar to what we did in class.
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [61]:
# Prompt Version 1 — Strict SQL Generator

sp_custom1 = """
### Instructions:
You are an expert SQL developer.

Your task is to convert a natural language question into a valid SQL query.

Rules:
- Only use tables and columns from the schema.
- Do not invent columns or tables.
- Return only SQL.
- Keep the query simple and efficient.
- Think carefully before generating the query.

### Database Schema

CREATE TABLE employees (
    ID_usr INT,
    name VARCHAR(100)
);

CREATE TABLE salary (
    ID_usr INT,
    salary INT,
    year INT
);

CREATE TABLE studies (
    ID_study INT,
    ID_usr INT,
    Institution VARCHAR(100),
    degree VARCHAR(100)
);

### Response
Based on your instructions, here is the SQL query I have generated to answer the question
  `{question}`:
  ```sql3
"""


In [62]:
import inspect
# Prompt Version 2 — Analyst Style Prompt

sp_custom2 = """
### Role
You are a data analyst working with a company HR database.

### Goal
Generate a SQL query to answer the user question.

### Important
- Understand relationships between tables before writing SQL.
- Use JOINs correctly.
- Use aggregation functions when necessary.
- Return only executable SQL code.

### Schema

CREATE TABLE employees (
    ID_usr INT,
    name VARCHAR(100)
);

CREATE TABLE salary (
    ID_usr INT,
    salary INT,
    year INT
);

CREATE TABLE studies (
    ID_study INT,
    ID_usr INT,
    Institution VARCHAR(100),
    degree VARCHAR(100)
);

### Example

Question: Return employee names and salaries.
```sql3
SELECT e.name, s.salary
FROM employees e
JOIN salary s
    ON e.ID_usr = s.ID_usr;
```
### Response
Based on your instructions, here is the SQL query I have generated to answer the question
  `{question}`:
  ```sql3
"""

In [63]:
# Prompt Version 3 — Step-by-Step Reasoning Prompt

sp_custom3 = """
### Instructions

Convert the natural language request into SQL.

Follow these steps internally:
1. Identify the required tables.
2. Identify relationships between tables.
3. Determine filtering conditions.
4. Determine aggregation or sorting needs.
5. Generate the final SQL query.

Only output the final SQL query.

### Schema

CREATE TABLE employees (
    ID_usr INT,
    name VARCHAR(100)
);

CREATE TABLE salary (
    ID_usr INT,
    salary INT,
    year INT
);

CREATE TABLE studies (
    ID_study INT,
    ID_usr INT,
    Institution VARCHAR(100),
    degree VARCHAR(100)
);

### Samples

Question: Return all employees.
```sql3
SELECT name
FROM employees;
```
### Response
Based on your instructions, here is the SQL query I have generated to answer the question
  `{question}`:
  ```sql3
"""

In [67]:
# Same question for all prompts

question = "Return the name of the best paid employee"

p1 = sp_custom1.format(question=question)
p2 = sp_custom2.format(question=question)
p3 = sp_custom3.format(question=question)


# ---------- Prompt 1 ----------

print("PROMPT 1\n")
print(p1)

input_sentences = tokenizer(p1, return_tensors="pt").to('cuda')

response = get_outputs(
    foundation_model,
    input_sentences,
    max_new_tokens=400
)

SQL = tokenizer.batch_decode(
    response,
    skip_special_tokens=True
)

torch.cuda.empty_cache()

print("\nSQL RESULT 1:\n")

print(
    SQL[0]
    .split("```sql3")[-1]
    .split("```")[0]
    .split(";")[0]
    .strip() + ";"
)


# ---------- Prompt 2 ----------

print("\n\nPROMPT 2\n")
print(p2)

input_sentences = tokenizer(p2, return_tensors="pt").to('cuda')

response = get_outputs(
    foundation_model,
    input_sentences,
    max_new_tokens=400
)

SQL = tokenizer.batch_decode(
    response,
    skip_special_tokens=True
)

torch.cuda.empty_cache()

print("\nSQL RESULT 2:\n")

print(
    SQL[0]
    .split("```sql3")[-1]
    .split("```")[0]
    .split(";")[0]
    .strip() + ";"
)


# ---------- Prompt 3 ----------

print("\n\nPROMPT 3\n")
print(p3)

input_sentences = tokenizer(p3, return_tensors="pt").to('cuda')

response = get_outputs(
    foundation_model,
    input_sentences,
    max_new_tokens=400
)

SQL = tokenizer.batch_decode(
    response,
    skip_special_tokens=True
)

torch.cuda.empty_cache()

print("\nSQL RESULT 3:\n")

print(
    SQL[0]
    .split("```sql3")[-1]
    .split("```")[0]
    .split(";")[0]
    .strip() + ";"
)

PROMPT 1


### Instructions:
You are an expert SQL developer.

Your task is to convert a natural language question into a valid SQL query.

Rules:
- Only use tables and columns from the schema.
- Do not invent columns or tables.
- Return only SQL.
- Keep the query simple and efficient.
- Think carefully before generating the query.

### Database Schema

CREATE TABLE employees (
    ID_usr INT,
    name VARCHAR(100)
);

CREATE TABLE salary (
    ID_usr INT,
    salary INT,
    year INT
);

CREATE TABLE studies (
    ID_study INT,
    ID_usr INT,
    Institution VARCHAR(100),
    degree VARCHAR(100)
);

### Response
Based on your instructions, here is the SQL query I have generated to answer the question
  `Return the name of the best paid employee`:
  ```sql3


SQL RESULT 1:

SELECT employees.name FROM employees JOIN (SELECT salary.id_usr, MAX(salary.salary) AS max_salary FROM salary GROUP BY salary.id_usr) AS subquery ON employees.id_usr = subquery.id_usr WHERE subquery.max_salary = (S

## Creative Prompt Results

### Prompt 1 — Strict SQL Generator

```sql
SELECT employees.name
FROM employees
JOIN (
    SELECT salary.id_usr, MAX(salary.salary) AS max_salary
    FROM salary
    GROUP BY salary.id_usr
) AS subquery
    ON employees.id_usr = subquery.id_usr
WHERE subquery.max_salary = (
    SELECT MAX(max_salary)
    FROM (
        SELECT MAX(salary.salary) AS max_salary
        FROM salary
        GROUP BY salary.id_usr
    ) AS subquery2
);
```

Prompt 1 produced a correct query, but it was more complicated than necessary. It used nested subqueries to find the maximum salary.

### Prompt 2 — Analyst Style Prompt

```sql
SELECT e.name, MAX(s.salary) AS max_salary
FROM employees e
JOIN salary s
    ON e.ID_usr = s.ID_usr
GROUP BY e.name
ORDER BY max_salary DESC
LIMIT 1;
```

Prompt 2 produced the best result. The query is simple, readable, and correctly returns the best paid employee.

### Prompt 3 — Step-by-Step Reasoning Prompt

```sql
SELECT employees.name, MAX(salary.salary) AS max_salary
FROM employees
JOIN salary
    ON employees.id_usr = salary.id_usr
GROUP BY employees.name
ORDER BY max_salary DESC NULLS LAST
LIMIT 1;
```
Prompt 3 also produced a correct result. It is slightly less clean than Prompt 2 because it does not use aliases, but the logic is correct.

## Conclusion
The creative prompts performed better than the earlier prompts. Prompt 1 was correct but too complex. Prompt 2 gave the cleanest and most professional SQL result. Prompt 3 was also correct and showed that step-by-step instructions can help the model understand the task better.

Overall, the best prompt was the analyst-style prompt because it included a clear role, a specific goal, important SQL rules, and one relevant example. This helped the model generate a correct and readable query.